# マテリアルズ・インフォマティクス入門ハンズオン

原子構造の作成から、DFT・経験力場・機械学習ポテンシャルによる計算、そして分子動力学（MD）までを一通り体験します。

**本日の流れ**

1. 構造作成（ASE）
2. GPAW で電子密度を確認
3. NaCl 格子定数スキャン（DFT）
4. 構造最適化（DFT）
5. Materials Project（構造最適化結果のデータベース）
6. 経験力場 EMT で高速に構造最適化
7. EMT で MD
8. CHGNet で MD

本編は完成コードを実行して進めます。**チャレンジ**は任意です。



## 環境確認

必要なパッケージが import できることを確認します。


In [ ]:
import ase
import asap3
import pymatgen
import nglview
import gpaw
import chgnet
import torch
import mp_api
import pandas as pd

print("imports ok")
print("torch cuda:", torch.cuda.is_available())


## 1. 構造作成

物質を原子レベルでデータとして扱うには、原子の種類と座標の情報を格納する必要があります。

ここでは、ASEパッケージの `Atoms` オブジェクトを使うことにしましょう。


### 1.1 水素分子 $\text{H}_2$

原子種と座標を直接指定して $\text{H}_2$ を作ります。


In [ ]:
from ase import Atoms
import nglview as nv

h2 = Atoms("H2", positions=[[0.0, 0.0, 0.0], [0.74, 0.0, 0.0]])
print(h2)
print("positions:\n", h2.positions)

view_h2 = nv.show_ase(h2)
view_h2.add_ball_and_stick()
view_h2

水素原子2つと、その座標がデータとして格納されました。

### チャレンジ 1（任意）：分子を作ってみましょう

好きな分子を、原子種と座標を指定して作ってみましょう。
下の例（直線状の $\text{CO}_2$）を書き換えてください。

In [ ]:
# 原子種と座標を書き換えてみましょう
my_molecule = Atoms("OCO", positions=[[0.0, 0.0, 0.0], [0.9, 0.0, 0.0], [1.8, 0.0, 0.0]])
print(my_molecule)
view_mol = nv.show_ase(my_molecule)
view_mol.add_ball_and_stick()
view_mol


### 1.2 水分子 $\text{H}_2\text{O}$

ASE の `molecule` 関数で登録済み分子を簡単に作れます。


In [ ]:
from ase.build import molecule

h2o = molecule("H2O")
print(h2o)
print("symbols :", h2o.get_chemical_symbols())
print("positions:\n", h2o.positions)

view_h2o = nv.show_ase(h2o)
view_h2o.add_ball_and_stick()
view_h2o


利用可能な分子名の一部は次のように確認できます。


In [ ]:
from ase.collections import g2

print("登録分子数:", len(g2.names))
print(g2.names)


### 1.3 NaCl 結晶（rocksalt）

`bulk` で岩石塩型 NaCl を作ります。`a` は立方格子の格子定数です（実験値はおよそ 5.64 Å）。


In [ ]:
from ase.build import bulk

nacl = bulk("NaCl", crystalstructure="rocksalt", a=5.64, cubic=True)
print(nacl)
print("cell lengths:", nacl.cell.lengths())
print("pbc:", nacl.pbc)

# 見やすくするため 2x2x2 超格子を表示
nacl_view = nacl * (2, 2, 2)
view_nacl = nv.show_ase(nacl_view)
view_nacl.add_unitcell()
view_nacl


## 2. 電子密度を見る

これまでは、原子と原子の間の結合距離、格子定数を天下り的に用いてきました。
原子の間の距離は、実際どのようにして決まるのか。
鍵となるのは電子です。

現代のシミュレーション手法を使えば、電子が空間にどう分布しているかを計算できます。

分子を第一原理計算でシミュレートし、電子密度を可視化してみましょう。

参考: [GPAW — All-electron density](https://gpaw.readthedocs.io/tutorialsexercises/wavefunctions/all_electron/all_electron_density.html)


In [ ]:
from pathlib import Path

from ase.build import molecule
from ase.io import write
from ase.units import Bohr
from gpaw import GPAW
import nglview as nv

Path("output").mkdir(exist_ok=True)

h2_dens = molecule("H2")  # 水素分子を作成
# h2_dens = Atoms("HOH", positions=[[0.0, 0.0, 0.0], [1.0, 0.0, 0.0], [1.5, 0.4, 0.0]])
h2_dens.center(vacuum=3.0)  # 真空の箱の中におく
v = nv.show_ase(h2_dens)
v.add_unitcell()
v

In [ ]:

h2_dens.calc = GPAW(
    mode="fd",
    h=0.25,          # Binder 向けに粗め（本番は 0.18 程度）
    xc="PBE",
    txt="output/h2_density.txt",
)
e = h2_dens.get_potential_energy()
print(f"H2 energy = {e:.4f} eV")

In [ ]:

# 全電子密度（Bohr^-3）。cube 書き出し時は Bohr^3 を掛けて Å^-3 相当にする慣例
density = h2_dens.calc.get_all_electron_density(gridrefinement=2)
cube_path = "output/h2_density.cube"
write(cube_path, h2_dens, data=density * Bohr**3)
print("wrote", cube_path)

view_dens = nv.show_ase(h2_dens)
view_dens.add_ball_and_stick()
view_dens.add_component(cube_path)
view_dens.clear_representations(component=1)
# isolevel は系・単位に依存するので、見づらい場合は値を調整してください
view_dens.add_surface(
    component=1,
    opacity=0.35,
    color="skyblue",
    isolevelType="value",
    isolevel=0.05,
)
view_dens

In [ ]:
forces = h2_dens.get_forces()
forces

In [ ]:
view_dens_force = view_dens
force_norms = np.linalg.norm(forces, axis=1)
print(f"原子数: {len(h2_dens)}")
print(f"最大の力 |F|_max  = {force_norms.max():.3f} eV/Å")
print(f"平均の力 |F|_mean = {force_norms.mean():.3f} eV/Å")
print("各原子の力 (eV/Å):")
for sym, f, fn in zip(h2_dens.get_chemical_symbols(), forces, force_norms):
    print(f"  {sym:2s}  F=({f[0]:+.3f}, {f[1]:+.3f}, {f[2]:+.3f})  |F|={fn:.3f}")

# 赤い矢印 = その原子にかかる力（長さは見やすさのためスケール）
force_scale = 0.15  # Å / (eV/Å)

for pos, f in zip(h2_dens.positions, forces):
    tip = pos + force_scale * f
    view_dens_force.shape.add_arrow(pos.tolist(), tip.tolist(), [1.0, 0.2, 0.2], 0.12)
view_dens_force

電子密度の等値面は、結合に電子が集まっている様子を直感的に示します。

このように、原子核の周りに雲のように分布する電子が、原子間の距離を決めることになります。

## 3. 第一原理計算による NaCl の格子定数スキャン

電子の状態を扱う理論計算によって、どのようにNaClの格子定数が決まるのかを体験しましょう。

「理系大学受験　化学の新研究」を参照すると、NaClの格子定数は2つの力の釣り合いで決まると書かれています。

- 引力に基づくエネルギー（正電荷・負電荷の引き合い）
- 斥力に基づくエネルギー（電子雲・原子核同士の反発）

この2つの力の釣り合いが、平衡格子定数を決めるというわけです。

第一原理計算ソフト **GPAW** を使い、実際にエネルギー的に安定な格子定数 $a$ が存在することを確認しましょう。
エネルギーが最小になる $a$ が、この計算条件での平衡格子定数の目安です。


In [ ]:
from pathlib import Path
import japanize_matplotlib
import numpy as np
import matplotlib.pyplot as plt
from ase.build import bulk
from gpaw import GPAW, PW

Path("output").mkdir(exist_ok=True)

# 実験値 5.64 Å 付近を粗くスキャン
a_list = np.linspace(4, 8.0, 20)
energies = []

for a in a_list:
    # 高速化のための粗い条件
    atoms = bulk("NaCl", crystalstructure="rocksalt", a=a)  # primitive (2 atoms)
    atoms.calc = GPAW(
        mode=PW(200),
        xc="PBE",
        kpts=(2, 2, 2),
        txt=f"output/nacl_a{a:.2f}.txt",
    )
    e = atoms.get_potential_energy()
    energies.append(e)
    print(f"a = {a:.3f} Å -> E = {e:.4f} eV  ({e / len(atoms):.4f} eV/atom)")

energies = np.array(energies)
a_min = a_list[np.argmin(energies)]
print(f"\n最小エネルギー付近の格子定数: a ≈ {a_min:.3f} Å")


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(a_list, energies / 2.0, "o-", label="GPAW PBE")
ax.axvline(5.64, color="gray", ls="--", label="実験値 ≈ 5.64 Å")
ax.set_xlabel("格子定数 a [Å]")
ax.set_ylabel("エネルギー [eV/atom]")
ax.set_title("NaCl rocksalt: E–a 曲線 (GPAW)")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig("output/nacl_Ea.png", dpi=120)
plt.show()


> **補足:** 本来の計算では平面波カットオフ・k点密度を十分大きくし、収束を確認します。
> ここではハンズオン用に意図的に軽い設定にしています。

格子定数スキャンは「1 パラメータを手で動かしてエネルギー曲面を見る」方法です。
次節では、力（と応力）を使って計算機に安定構造を探させる **構造最適化** に進みます。


## 4. 構造最適化（DFT / GPAW）

構造最適化（構造緩和）は、原子にかかる力 $\mathbf{F}_i = -\partial E / \partial \mathbf{R}_i$（とセルに対する応力）が小さくなるよう座標・格子を更新し、局所安定構造を求める計算です。

格子定数スキャンでは $a$ を手動で変えましたが、最適化ではアルゴリズム（例: BFGS）が自動でパラメータを動かします。

まず、理想的な岩塩構造の原子をランダムにずらした「無理な初期構造」を作り、各原子にかかる力を矢印で可視化します。
その後、同じ構造を第一原理計算で緩和します。

In [ ]:
from pathlib import Path

import numpy as np
from ase.build import bulk
from gpaw import GPAW, PW
import nglview as nv

Path("output").mkdir(exist_ok=True)

# 理想的な岩塩構造を作り、原子をランダムに変位させる
nacl_opt = bulk("NaCl", crystalstructure="rocksalt", a=5.64, cubic=True)
nacl_opt.rattle(stdev=0.25, seed=42)  # 標準偏差 0.25 Å

nacl_opt.calc = GPAW(
    mode=PW(200),
    xc="PBE",
    kpts=(2, 2, 2),
    txt="output/nacl_opt.txt",
)

forces = nacl_opt.get_forces()
force_norms = np.linalg.norm(forces, axis=1)
print(f"原子数: {len(nacl_opt)}")
print(f"最大の力 |F|_max  = {force_norms.max():.3f} eV/Å")
print(f"平均の力 |F|_mean = {force_norms.mean():.3f} eV/Å")
print("各原子の力 (eV/Å):")
for sym, f, fn in zip(nacl_opt.get_chemical_symbols(), forces, force_norms):
    print(f"  {sym:2s}  F=({f[0]:+.3f}, {f[1]:+.3f}, {f[2]:+.3f})  |F|={fn:.3f}")

# 赤い矢印 = その原子にかかる力（長さは見やすさのためスケール）
force_scale = 1.5  # Å / (eV/Å)
view_f = nv.show_ase(nacl_opt)
view_f.add_unitcell()
for pos, f in zip(nacl_opt.positions, forces):
    tip = pos + force_scale * f
    view_f.shape.add_arrow(pos.tolist(), tip.tolist(), [1.0, 0.2, 0.2], 0.12)
view_f


In [ ]:
from ase.filters import FrechetCellFilter
from ase.io import Trajectory, write
from ase.optimize import BFGS

print(f"初期 cell lengths: {nacl_opt.cell.lengths()}")
print(f"初期エネルギー: {nacl_opt.get_potential_energy():.4f} eV")

fcf = FrechetCellFilter(nacl_opt)
opt = BFGS(fcf, logfile="output/nacl_gpaw_opt.log")
# Filter ではなく元の atoms を書く（各フレームの座標・力が正しく残る）
traj_writer = Trajectory("output/nacl_gpaw_opt.traj", "w", nacl_opt)
opt.attach(traj_writer)
opt.run(fmax=0.05)
traj_writer.close()

print(f"最適化後 cell lengths: {nacl_opt.cell.lengths()}")
print(f"最適化後のエネルギー: {nacl_opt.get_potential_energy():.4f} eV")
print(f"conventional a ≈ {nacl_opt.cell.lengths().mean():.3f} Å（実験値 ≈ 5.64 Å）")
write("output/nacl_gpaw_opt.cif", nacl_opt)


### 4.1 緩和軌跡の可視化

エネルギーと力の推移をプロットし、構造の変化を nglview で再生します。
スライダーでフレームを選ぶと、その時点の力ベクトル（赤矢印）も表示できます。


In [ ]:
from IPython.display import clear_output, display

import japanize_matplotlib
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import nglview as nv
from ase.io import read

traj = read("output/nacl_gpaw_opt.traj", index=":")
energies = [atoms.get_potential_energy() for atoms in traj]
fmax_list = [np.linalg.norm(atoms.get_forces(), axis=1).max() for atoms in traj]
fmean_list = [np.linalg.norm(atoms.get_forces(), axis=1).mean() for atoms in traj]
steps = np.arange(len(traj))

print(f"frames: {len(traj)}")
print(f"E: {energies[0]:.4f} → {energies[-1]:.4f} eV")
print(f"|F|_max: {fmax_list[0]:.3f} → {fmax_list[-1]:.3f} eV/Å")

fig, ax1 = plt.subplots(figsize=(6, 3.5))
ax1.plot(steps, energies, "C0-o", ms=4, label="Energy (eV)")
ax1.set_xlabel("緩和ステップ")
ax1.set_ylabel("Energy (eV)", color="C0")
ax1.tick_params(axis="y", labelcolor="C0")

ax2 = ax1.twinx()
ax2.plot(steps, fmax_list, "C1-s", ms=4, label="|F|_max")
ax2.plot(steps, fmean_list, "C1--", alpha=0.7, label="|F|_mean")
ax2.set_ylabel("Force (eV/Å)", color="C1")
ax2.tick_params(axis="y", labelcolor="C1")

ax1.set_title("NaCl GPAW 構造最適化")
ax1.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

# 構造トラジェクトリ（再生・スクラブ）
view_traj = nv.show_asetraj(traj, gui=True)
view_traj.add_unitcell()
display(view_traj)

# フレームごとの力ベクトル（赤矢印）
force_scale = 1.5  # Å / (eV/Å)
slider = widgets.IntSlider(value=0, min=0, max=len(traj) - 1, description="frame")
out = widgets.Output()


def show_forces(change=None):
    i = slider.value
    atoms = traj[i]
    forces = atoms.get_forces()
    fmax = np.linalg.norm(forces, axis=1).max()
    with out:
        clear_output(wait=True)
        print(
            f"frame {i}: E={atoms.get_potential_energy():.4f} eV, "
            f"|F|_max={fmax:.3f} eV/Å"
        )
        view_f = nv.show_ase(atoms)
        view_f.add_unitcell()
        for pos, f in zip(atoms.positions, forces):
            tip = pos + force_scale * f
            view_f.shape.add_arrow(pos.tolist(), tip.tolist(), [1.0, 0.2, 0.2], 0.12)
        display(view_f)


slider.observe(show_forces, names="value")
display(slider, out)
show_forces()


力が閾値（`fmax`）以下になると収束です。ランダムにずらした原子も力の向きに従って格子点付近へ戻り、エネルギーと力は単調に小さくなっていきます（計算条件が粗いと実験値とのずれは残ります）。

Materials Project などの公開構造も、基本的にこうした DFT による構造最適化の結果です。


## 5. Materials Project から構造を取得

[Materials Project](https://materialsproject.org/) は、DFT による構造最適化・物性計算の結果を大規模に蓄積したデータベースです。
前節で手元の NaCl を 1 件緩和しましたが、Materials Project では同様の計算が結晶に対して網羅的に行われ、組成・安定性・バンドギャップなどの条件で検索できます。

Python クライアント `mp-api` の `MPRester` を使います。

参考: [Materials Project Workshop — Materials API](https://workshop.materialsproject.org/lessons/04_materials_api/MAPI%20Lesson%20(filled)/)、
[公式ドキュメント](https://docs.materialsproject.org/downloading-data/using-the-api/getting-started)


### 5.1 API キーの準備

1. [Materials Project](https://next-gen.materialsproject.org/) にログイン
2. 右上の **API** から API キーを取得



In [ ]:
import os

# どちらか一方を設定:
# 1) 環境変数 MP_API_KEY（推奨）
# 2) 下の文字列に自分のキーを入れる（ローカル作業時のみ）
MP_API_KEY = os.environ.get("MP_API_KEY", "cd3eAs8AWW1VBgNNhn5iA6Ws0c3Sgw21")

if not MP_API_KEY:
    # 例: MP_API_KEY = "your_api_key_here"
    raise ValueError(
        "MP_API_KEY が未設定です。"
        "https://next-gen.materialsproject.org/api でキーを取得し、"
        "環境変数かこのセルで設定してください。"
    )

print("API key: 設定済み（先頭4文字）=", MP_API_KEY[:4], "...")


### 5.2 material_id で構造を取得

Materials Project の各物質には `mp-XXXX` 形式の ID があります。
例: ダイヤモンド型 Si は `mp-149`、岩塩型 NaCl は `mp-22862`。

`get_structure_by_material_id` が最も手軽です。取得できる構造は、あらかじめ DFT で緩和されたものだと考えてください。


In [ ]:
from mp_api.client import MPRester
import nglview as nv
from pymatgen.io.ase import AseAtomsAdaptor

with MPRester(MP_API_KEY) as mpr:
    # Si (diamond): mp-149
    si_structure = mpr.get_structure_by_material_id("mp-149")

print(si_structure)
print("formula:", si_structure.composition.reduced_formula)
print("lattice a,b,c:", si_structure.lattice.abc)

si_atoms = AseAtomsAdaptor.get_atoms(si_structure)
view_si = nv.show_ase(si_atoms)
view_si.add_unitcell()
view_si


### 5.3 組成・化学系で検索する

`mpr.materials.summary.search(...)` を使います。

- `formula="NaCl"` … 組成式で検索
- `chemsys="Na-Cl"` … 化学系（元素の組み合わせ）で検索
- `fields=[...]` … 返す項目を絞ると高速
- `energy_above_hull=(0, 0.05)` … 熱力学的に安定に近いものだけ、など


In [ ]:
with MPRester(MP_API_KEY) as mpr:
    docs = mpr.materials.summary.search(
        formula="NaCl",
        fields=[
            "material_id",
            "formula_pretty",
            "structure",
            "energy_above_hull",
            "band_gap",
            "symmetry",
        ],
    )

print(f"NaCl 組成のヒット数: {len(docs)}")
for doc in docs[:5]:
    sg = doc.symmetry.symbol if doc.symmetry else "?"
    print(
        f"{doc.material_id}  {doc.formula_pretty:8s}  "
        f"E_hull={doc.energy_above_hull:.4f} eV/atom  "
        f"Eg={doc.band_gap:.3f} eV  SG={sg}"
    )


安定相（`energy_above_hull` が小さいもの）の構造を取り出し、ASE に変換して可視化します。


In [ ]:
# energy_above_hull が最小のものを採用
docs_sorted = sorted(docs, key=lambda d: d.energy_above_hull)
best = docs_sorted[0]
print("選択:", best.material_id, best.formula_pretty, f"E_hull={best.energy_above_hull:.6f}")

nacl_mp = best.structure
nacl_atoms_mp = AseAtomsAdaptor.get_atoms(nacl_mp)
print(nacl_atoms_mp)
print("cell lengths:", nacl_atoms_mp.cell.lengths())

view_nacl_mp = nv.show_ase(nacl_atoms_mp * (2, 2, 2))
view_nacl_mp.add_unitcell()
view_nacl_mp


### 5.4 化学系での検索例（Si–O）

Workshop と同様に、化学系 `Si-O` で候補を絞り、物性条件を付けてスクリーニングできます。


In [ ]:
with MPRester(MP_API_KEY) as mpr:
    sio_docs = mpr.materials.summary.search(
        chemsys="Si-O",
        energy_above_hull=(0.0, 0.05),  # 比較的安定
        fields=["material_id", "formula_pretty", "energy_above_hull", "band_gap", "nsites"],
    )

print(f"Si-O 系（E_hull ≤ 50 meV/atom）: {len(sio_docs)} 件")
for doc in sorted(sio_docs, key=lambda d: d.energy_above_hull)[:8]:
    print(
        f"{doc.material_id}  {doc.formula_pretty:10s}  "
        f"nsites={doc.nsites:3d}  "
        f"E_hull={doc.energy_above_hull:.4f}  Eg={doc.band_gap:.3f}"
    )


### チャレンジ 2（任意）：検索条件を変えてみる

上の Si–O の例を参考に、**別の化学系**や **物性条件**で検索してみましょう。
時間がなければ飛ばして構いません。

例:
- `chemsys="Al-O"` や `chemsys="Li-O"`
- `band_gap=(1.0, 3.0)` を追加して半導体っぽいものだけにする
- `energy_above_hull=(0.0, 0.01)` でより安定なものだけにする


In [ ]:
# TODO: chemsys や条件を書き換えて実行
with MPRester(MP_API_KEY) as mpr:
    my_docs = mpr.materials.summary.search(
        chemsys="Al-O",  # ← ここを変えてみる
        energy_above_hull=(0.0, 0.05),
        # band_gap=(1.0, 3.0),  # ← 必要ならコメントを外す
        fields=["material_id", "formula_pretty", "energy_above_hull", "band_gap", "nsites"],
    )

print(f"ヒット数: {len(my_docs)} 件")
for doc in sorted(my_docs, key=lambda d: d.energy_above_hull)[:8]:
    print(
        f"{doc.material_id}  {doc.formula_pretty:10s}  "
        f"nsites={doc.nsites:3d}  "
        f"E_hull={doc.energy_above_hull:.4f}  Eg={doc.band_gap:.3f}"
    )


取得した構造は「すでに DFT で最適化された初期構造」として、以降の経験力場・ML ポテンシャル・MD に渡せます。
手元でゼロから DFT 緩和するコストを大幅に省略できるのが、Materials Project を使う利点です。


## 6. 経験力場 EMT による高速な構造最適化

DFT は正確ですが遅いです。金属向けの経験力場 **EMT**（Effective Medium Theory, ASAP 実装）なら、同じ構造最適化が桁違いに速く終わります。

ここではセルをやや膨らませ、原子位置を少し乱した fcc-Al を BFGS で緩和します。DFT の近似として「速さ優先」の計算手段を使う、という位置づけです。


In [ ]:
from pathlib import Path

import numpy as np
from asap3 import EMT
from ase.build import bulk
from ase.optimize import BFGS
import nglview as nv

Path("output").mkdir(exist_ok=True)

atoms = bulk("Al", "fcc", a=4.05, cubic=True)
atoms *= (2, 2, 2)

# 原子を微小変位させて「未緩和」状態を作る
rng = np.random.default_rng(0)
atoms.positions += rng.normal(scale=0.08, size=atoms.positions.shape)

atoms.calc = EMT()
e0 = atoms.get_potential_energy()
f0 = np.linalg.norm(atoms.get_forces(), axis=1).max()
print(f"緩和前: E = {e0:.4f} eV,  max|F| = {f0:.4f} eV/Å")

opt = BFGS(atoms, trajectory="output/al_emt_opt.traj", logfile="output/al_emt_opt.log")
opt.run(fmax=0.01)

e1 = atoms.get_potential_energy()
f1 = np.linalg.norm(atoms.get_forces(), axis=1).max()
print(f"緩和後: E = {e1:.4f} eV,  max|F| = {f1:.4f} eV/Å")
print(f"cell lengths: {atoms.cell.lengths()}")

view_al_opt = nv.show_ase(atoms)
view_al_opt.add_unitcell()
view_al_opt


EMT は適用できる元素・現象が限られます（金属向きで、酸化物や任意組成には不向き）。
汎用性が必要なら後述の CHGNet のような機械学習ポテンシャル、精度が必要なら DFT に戻る、という使い分けになります。

次は、同じ EMT で有限温度のダイナミクス（MD）を回します。


## 7. ASAP (EMT) による Al の MD

[ASAP](https://wiki.fysik.dtu.dk/asap/) の **EMT** 力場は、金属の高速な古典 MD に便利です。
ここでは fcc-Al の NVE（Velocity Verlet）シミュレーションを短時間実行します。

参考: [Atomistic Simulation Tutorial (Matlantis)](https://docs.matlantis.com/atomistic-simulation-tutorial/ja/index.html)


In [ ]:
import os
from pathlib import Path
from time import perf_counter

from asap3 import EMT
from ase.build import bulk
from ase import units
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.md.verlet import VelocityVerlet
from ase.md import MDLogger

Path("output").mkdir(exist_ok=True)

atoms = bulk("Al", "fcc", a=4.05, cubic=True)
atoms *= (3, 3, 3)  # 108 atoms
atoms.pbc = True
atoms.calc = EMT()

time_step = 1.0       # fs
temperature = 800     # K
num_md_steps = 20000
num_interval = 100

MaxwellBoltzmannDistribution(atoms, temperature_K=temperature, force_temp=True)
Stationary(atoms)

traj_file = "output/al_asap_nve.traj"
log_file = "output/al_asap_nve.log"
for path in (traj_file, log_file):
    if os.path.exists(path):
        os.remove(path)

dyn = VelocityVerlet(
    atoms,
    time_step * units.fs,
    trajectory=traj_file,
    loginterval=num_interval,
)

temperatures = []
times_fs = []

def record():
    step = dyn.get_number_of_steps()
    temperatures.append(atoms.get_temperature())
    times_fs.append(step * time_step)
    if step % (num_interval * 5) == 0:
        print(
            f"step={step:5d}  "
            f"Etot={atoms.get_total_energy():.4f} eV  "
            f"T={atoms.get_temperature():.1f} K"
        )

dyn.attach(record, interval=num_interval)
dyn.attach(MDLogger(dyn, atoms, log_file, header=True, stress=False, peratom=False, mode="w"), interval=num_interval)

t0 = perf_counter()
print("ASAP EMT / Al NVE MD 開始")
dyn.run(num_md_steps)
print(f"完了: {perf_counter() - t0:.1f} s")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nglview as nv
from ase.io import read

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(np.array(times_fs) / 1000.0, temperatures, "-")
ax.set_xlabel("時間 [ps]")
ax.set_ylabel("温度 [K]")
ax.set_title("fcc-Al NVE (ASAP EMT)")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

traj = read(traj_file, index=":")
print(f"trajectory frames: {len(traj)}")

v = nv.show_asetraj(traj, gui=True)
v.add_unitcell()
v

NVE では総エネルギーがほぼ保存され、温度は目標値の周りで揺らぎます。
温度を厳密に制御したい場合は NVT（例: `NVTBerendsen`）を使います。


### チャレンジ 3（任意）：温度とステップ数を変えてみる

上の Al MD を、パラメータだけ変えて短く再実行してみましょう。
結果の温度グラフやトラジェクトリがどう変わるか見てください。やらなくても次へ進んで大丈夫です。


In [ ]:
# TODO: temperature と num_md_steps を変えて再実行（短め推奨）
import os
import numpy as np
import matplotlib.pyplot as plt
import nglview as nv
from asap3 import EMT
from ase import units
from ase.build import bulk
from ase.io import read
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.md.verlet import VelocityVerlet

temperature = 300      # ← 例: 300, 800, 1200
num_md_steps = 2000    # ← 本編は 20000。チャレンジでは短くてOK
time_step = 1.0        # fs
num_interval = 100

atoms = bulk("Al", "fcc", a=4.05, cubic=True)
atoms *= (3, 3, 3)
atoms.pbc = True
atoms.calc = EMT()

MaxwellBoltzmannDistribution(atoms, temperature_K=temperature, force_temp=True)
Stationary(atoms)

traj_file = "output/al_asap_nve_challenge.traj"
log_file = "output/al_asap_nve_challenge.log"
for path in (traj_file, log_file):
    if os.path.exists(path):
        os.remove(path)

dyn = VelocityVerlet(
    atoms,
    time_step * units.fs,
    trajectory=traj_file,
    loginterval=num_interval,
)

temperatures = []
times_fs = []

def record():
    step = dyn.get_number_of_steps()
    temperatures.append(atoms.get_temperature())
    times_fs.append(step * time_step)

dyn.attach(record, interval=num_interval)
dyn.run(num_md_steps)
print(f"完了: T_init={temperature} K, steps={num_md_steps}")

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(np.array(times_fs) / 1000.0, temperatures, "-")
ax.set_xlabel("時間 [ps]")
ax.set_ylabel("温度 [K]")
ax.set_title(f"fcc-Al NVE (T={temperature} K)")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

traj = read(traj_file, index=":")
v = nv.show_asetraj(traj, gui=True)
v.add_unitcell()
v


## 8. CHGNet による MD

上記のように、特定の系に特化した経験式で、原子間の相互作用を表すことができます。
このような経験ポテンシャルは第一原理計算よりも高速です。

一方で、第一原理計算は、低速ですが、より幅広い系に使えます。

では、高速なポテンシャルを幅広い系に使うには？

[CHGNet](https://chgnet.lbl.gov/) は結晶の Universal Machine Learning Potential です。
DFT よりはるかに速く、多様な組成に適用できます（EMT が苦手な系にも使えます）。

ここでは Li の BCC 構造に対し、短い NVT MD を実行します。


In [ ]:
import os
from pathlib import Path
from time import perf_counter

from ase.build import bulk
from chgnet.model import CHGNet
from chgnet.model.dynamics import MolecularDynamics

Path("output").mkdir(exist_ok=True)

atoms = bulk("Li", "bcc", a=3.51, cubic=True)
atoms *= (2, 2, 2)

chgnet_model = CHGNet.load()

traj_file = "output/li_chgnet_nvt.traj"
log_file = "output/li_chgnet_nvt.log"
for path in (traj_file, log_file):
    if os.path.exists(path):
        os.remove(path)

md = MolecularDynamics(
    atoms=atoms,
    model=chgnet_model,
    ensemble="nvt",
    thermostat="Berendsen",
    temperature=300,
    timestep=2,  # fs
    trajectory=traj_file,
    logfile=log_file,
    loginterval=10,
    use_device="cpu",
)

print("CHGNet NVT MD 開始")
t0 = perf_counter()
md.run(200)  # 0.4 ps
print(f"完了: {perf_counter() - t0:.1f} s")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nglview as nv
from ase.io import read

traj = read(traj_file, index=":")
print(f"frames = {len(traj)}")
print(f"最終構造: {traj[-1]}")

with open(log_file) as f:
    header = f.readline().strip()
print("log header:", header)

log = np.loadtxt(log_file, skiprows=1)
if log.ndim == 1:
    log = log.reshape(1, -1)

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(log[:, 0], log[:, -1], "-")
ax.set_xlabel("時間 [ps]")
ax.set_ylabel("温度 [K]")
ax.set_title("Li BCC NVT (CHGNet)")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

view_li = nv.show_ase(traj[-1])
view_li.add_unitcell()
view_li


### 単点エネルギーの例

MD の前に、CHGNet Calculator で単点計算もできます。


In [ ]:
from ase.build import bulk
from chgnet.model.dynamics import CHGNetCalculator

atoms_sp = bulk("Li", "bcc", a=3.51)
atoms_sp.calc = CHGNetCalculator(model=chgnet_model, use_device="cpu")
print(f"Li BCC energy = {atoms_sp.get_potential_energy():.4f} eV")


## 9. CHGNet 構造最適化

CHGNet の `StructOptimizer` で構造緩和し、軌跡を **nglview** と matplotlib で可視化します。
例題は Materials Project の LiMnO₂（[mp-18767](https://materialsproject.org/materials/mp-18767)）です。


### 9.1 構造の読み込みと摂動

平衡構造を少し乱し、セルも膨らませてから緩和します。


In [ ]:
import numpy as np
from pymatgen.core import Structure

structure = Structure.from_file("input/mp-18767-LiMnO2.cif")
print("original:", structure.get_space_group_info())

# 原子座標を微小摂動
rng = np.random.default_rng(0)
for site in structure:
    site.coords += rng.normal(size=3) * 0.3

# セル体積を 10% 拡大
structure.scale_lattice(structure.volume * 1.1)
print("perturbed:", structure.get_space_group_info())
print(structure)


### 9.2 StructOptimizer で緩和

`FIRE` オプティマイザで力と応力が小さくなるまで緩和します（Binder では `use_device="cpu"`）。


In [ ]:
import pandas as pd
from chgnet.model import StructOptimizer

relaxer = StructOptimizer(use_device="cpu")
result = relaxer.relax(structure, fmax=0.1, steps=200, verbose=True)
trajectory = result["trajectory"]

print("final structure:")
print(result["final_structure"])
print(f"final energy = {trajectory.energies[-1]:.4f} eV")


In [ ]:
e_col = "Energy (eV)"
force_col = "Force (eV/Å)"
df_traj = pd.DataFrame(trajectory.energies, columns=[e_col])
df_traj[force_col] = [
    np.linalg.norm(force, axis=1).mean()
    for force in trajectory.forces
]
df_traj.index.name = "step"
df_traj.tail()


### 9.3 緩和軌跡の可視化（nglview）

エネルギー・力の推移をプロットし、緩和過程の構造を nglview のトラジェクトリとして再生します。


In [ ]:
import japanize_matplotlib
import matplotlib.pyplot as plt
import nglview as nv
from ase import Atoms

# エネルギー・平均力の推移
fig, ax1 = plt.subplots(figsize=(6, 3.5))
ax1.plot(df_traj.index, df_traj[e_col], "C0-", label=e_col)
ax1.axhline(-59.09, color="C0", ls=":", alpha=0.7, label="DFT final (MP)")
ax1.set_xlabel("緩和ステップ")
ax1.set_ylabel(e_col, color="C0")
ax1.tick_params(axis="y", labelcolor="C0")

ax2 = ax1.twinx()
ax2.plot(df_traj.index, df_traj[force_col], "C1-", label=force_col)
ax2.set_ylabel(force_col, color="C1")
ax2.tick_params(axis="y", labelcolor="C1")

ax1.set_title("LiMnO₂ (mp-18767) CHGNet 緩和")
ax1.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

# CHGNet trajectory → ASE Atoms 列へ変換して nglview 表示
symbols = [str(site.specie) for site in structure]
traj_ase = [
    Atoms(
        symbols=symbols,
        positions=trajectory.atom_positions[i],
        cell=trajectory.cells[i],
        pbc=True,
    )
    for i in range(len(trajectory))
]
print(f"relaxation frames: {len(traj_ase)}")

view_relax = nv.show_asetraj(traj_ase, gui=True)
view_relax.add_unitcell()
view_relax


## 10. 金属表面での酸化反応（CHGNet）

参考: [Atomistic Simulation Tutorial 6.2 — 計算事例３：金属表面での酸化反応](https://docs.matlantis.com/atomistic-simulation-tutorial/ja/6_2_md-nvt.html)

きれいなfcc型のAl金属の表面が酸素雰囲気下にさらされた状態を考えます。
アルミニウムは室温でも酸化しやすいので、NVT-MD でその様子を再現してみましょう。



In [ ]:
from pathlib import Path

import nglview as nv
from ase.io import read

Path("output").mkdir(exist_ok=True)

atoms = read("input/fcc111_Al_3x4x6_vac10A_20O2.cif")
atoms.pbc = True
print(atoms)
print({s: list(atoms.symbols).count(s) for s in sorted(set(atoms.symbols))})

view0 = nv.show_ase(atoms)
view0.add_unitcell()
view0




金属表面と気相分子の反応を扱うには、O を含む系に対応したポテンシャルが必要です（ASAP の EMT では不向きです）。

300 K・NVT（Nosé–Hoover）で短い MD を実行します。


### Nosé–Hoover 熱浴

ASE では `NPT` クラスに `pfactor=None` を渡すと、体積一定の **Nosé–Hoover NVT** になります。

- https://wiki.fysik.dtu.dk/ase/ase/md.html#nose-hoover-dynamics

時定数 `ttime`（$\tau_T$）はおよそ 20–25 fs が目安です。小さすぎると不安定、大きすぎると温度収束が遅くなります。


In [ ]:
import os
from time import perf_counter

from ase import units
from ase.md import MDLogger
from ase.md.npt import NPT
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from chgnet.model import CHGNet
from chgnet.model.dynamics import CHGNetCalculator

chgnet_model = CHGNet.load()
atoms.calc = CHGNetCalculator(model=chgnet_model, use_device="cpu")

time_step = 1.0       # fs
temperature = 300     # K
num_md_steps = 200    # Binder 向けテスト。本格計算は 10000 程度
num_interval = 50
ttime = 20.0          # thermostat time constant [fs]

traj_file = "output/al111_o2_nvt_nosehoover.traj"
log_file = "output/al111_o2_nvt_nosehoover.log"
for path in (traj_file, log_file):
    if os.path.exists(path):
        os.remove(path)

MaxwellBoltzmannDistribution(atoms, temperature_K=temperature, force_temp=True)
Stationary(atoms)

dyn = NPT(
    atoms,
    time_step * units.fs,
    temperature_K=temperature,
    externalstress=0.1e-6 * units.GPa,  # NVT では実質無視
    ttime=ttime * units.fs,
    pfactor=None,  # None → NVT (Nosé–Hoover)
    loginterval=num_interval,
    trajectory=traj_file,
)

def print_dyn():
    step = dyn.get_number_of_steps()
    print(
        f"step={step:5d}  "
        f"Etot={atoms.get_total_energy():.3f} eV  "
        f"T={atoms.get_temperature():.1f} K"
    )

dyn.attach(print_dyn, interval=num_interval)
dyn.attach(
    MDLogger(dyn, atoms, log_file, header=True, stress=True, peratom=True, mode="w"),
    interval=num_interval,
)

print("Al(111)+O2 Nosé–Hoover NVT (CHGNet) 開始")
t0 = perf_counter()
dyn.run(num_md_steps)
print(f"完了: {perf_counter() - t0:.1f} s")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ase.io import read

traj = read(traj_file, index=":")
print(f"frames = {len(traj)}")

with open(log_file) as f:
    header = f.readline().strip()
print("header:", header)

log = np.loadtxt(log_file, skiprows=1)
if log.ndim == 1:
    log = log.reshape(1, -1)

# ASE MDLogger: Time[ps] ... T[K] は通常5列目 (index 4)
time_ps = log[:, 0]
temp_K = log[:, 4] if log.shape[1] > 4 else log[:, -1]

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(time_ps, temp_K, "-")
ax.set_xlabel("時間 [ps]")
ax.set_ylabel("温度 [K]")
ax.set_title("Al(111)+O₂ NVT (CHGNet / Nosé–Hoover)")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

view1 = nv.show_ase(traj[-1])
view1.add_unitcell()
view1


短いテスト計算でも、O₂ が表面近傍へ近づく様子が見えることがあります。
本格的な時間スケール（数 ps〜）では、チュートリアルと同様に **吸着 → O–O 開裂 → 表面酸化** が進みます。

アルミニウムは室温・低酸素分圧でも酸化物を作りやすいため、狭いセルに多数の O₂ を詰めた条件では酸化が起きやすいです（[Ellingham diagram](https://en.wikipedia.org/wiki/Ellingham_diagram) も参照）。

<figure style="text-align: center">
<img src="assets/Fig6-2_O2_adsorption_on_fcc111_Al.png" width="520"/>
<figcaption>O₂ の吸着・開裂プロセスの模式（チュートリアル Fig.6-2i）</figcaption>
</figure>

このように NVT-MD では、ガスと固体表面の反応ダイナミクスを原子レベルで追跡できます。


## まとめ

| 内容 | ツール | ポイント |
|------|--------|----------|
| H₂ / H₂O / NaCl の作成 | ASE (`Atoms`, `molecule`, `bulk`) | 構造はすべての計算の出発点 |
| 電子密度の可視化 | GPAW + nglview | DFT が扱う電子分布を直感的に確認 |
| NaCl 格子定数スキャン | GPAW (DFT) | $E(a)$ から平衡格子定数を見積もる |
| NaCl 構造最適化 | GPAW + BFGS（セル緩和） | 力の可視化と緩和軌跡（エネルギー・力・構造） |
| データベースから構造取得 | Materials Project (`mp-api`) | DFT 緩和済み構造の大規模 DB |
| Al の高速構造最適化 | ASAP EMT + BFGS | 経験力場による近似・高速化 |
| Al の MD | ASAP EMT | 金属向けの高速古典力場 |
| Li の MD | CHGNet | 汎用 ML ポテンシャル |
| LiMnO₂ 構造最適化 | CHGNet `StructOptimizer` + nglview | 緩和軌跡の可視化 |
| Al(111) 表面酸化 | CHGNet + Nosé–Hoover NVT | 吸着・解離・酸化のダイナミクス |

チャレンジは任意です（分子作成・MP検索条件・MDの温度/ステップ）。本編を優先し、余った時間で試してください。

次のステップとしては、NVT/NPT アンサンブル、表面・欠陥モデルなどが挙げられます。
詳細は [Atomistic Simulation Tutorial](https://docs.matlantis.com/atomistic-simulation-tutorial/ja/index.html)、
[CHGNet examples](https://github.com/CederGroupHub/chgnet/tree/main/examples)、
[Materials Project Workshop](https://workshop.materialsproject.org/) を参照してください。
